# zero-shot-image-classification

In [ ]:
from pathlib import Path
from transformers import pipeline

root = Path("..")
image = str(root / "dataset/images/photo5.jpg")
model_dir = str(root / "weights/clip-vit-base-patch32")

clf = pipeline(
    task="zero-shot-image-classification",
    model=model_dir,
    device="mps",
    local_files_only=True,
)

labels = [
    "reinforced concrete building frame",
    "brick or block wall masonry",
    "facade cladding in progress",
    "scaffolding on a building",
    "excavation pit and earthworks",
    "construction site trailers and containers",
    "tower crane on construction site",
    "window glazing",
]
clf(image, candidate_labels=labels)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 45490.87it/s]


[{'score': 0.720207691192627,
  'label': 'construction site trailers and containers'},
 {'score': 0.19798780977725983, 'label': 'facade cladding in progress'},
 {'score': 0.06212330609560013, 'label': 'tower crane on construction site'},
 {'score': 0.012310934253036976,
  'label': 'reinforced concrete building frame'},
 {'score': 0.004158010706305504, 'label': 'excavation pit and earthworks'},
 {'score': 0.0019858554005622864, 'label': 'scaffolding on a building'},
 {'score': 0.0008591037476435304, 'label': 'brick or block wall masonry'},
 {'score': 0.0003673168830573559, 'label': 'window glazing'}]

In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(1234)

root = Path("..")
image_path = str(root / "dataset/images/photo4.jpg")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-VL", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen-VL",
    device_map="mps",
    trust_remote_code=True,
).eval()

query = tokenizer.from_list_format([
    {"image": image_path},
    {"text": "Generate the caption in English with grounding:"},
])
inputs = tokenizer(query, return_tensors="pt")
inputs = inputs.to(model.device)
pred = model.generate(**inputs)
response = tokenizer.decode(pred.cpu()[0], skip_special_tokens=False)
print(response)

image = tokenizer.draw_bbox_on_latest_picture(response)
if image:
    image.save(str(root / "dataset/images/photo4_grounded.jpg"))
else:
    print("no box")
